In [2]:
!apt-get install -y wget

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wget is already the newest version (1.21.2-2ubuntu1.1).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


In [6]:
!wget --content-disposition "https://cernbox.cern.ch/remote.php/dav/public-files/zUvpkKhXIp0MJ0g/top_gun_opendata_0.parquet"

--2026-03-26 14:46:05--  https://cernbox.cern.ch/remote.php/dav/public-files/zUvpkKhXIp0MJ0g/top_gun_opendata_0.parquet
Resolving cernbox.cern.ch (cernbox.cern.ch)... 128.142.53.35, 128.142.53.28, 137.138.120.151, ...
Connecting to cernbox.cern.ch (cernbox.cern.ch)|128.142.53.35|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1171158319 (1.1G) [application/octet-stream]
Saving to: ‘top_gun_opendata_0.parquet.1’

top_gun_opendata_0. 100%[===================>]   1.09G  1.88MB/s    in 10m 32s 

2026-03-26 14:56:37 (1.77 MB/s) - ‘top_gun_opendata_0.parquet.1’ saved [1171158319/1171158319]



In [8]:
!pip install pyarrow

In [10]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("top_gun_opendata_0.parquet.1")
batch = parquet_file.read_row_group(0)
df = batch.to_pandas()

df = df.iloc[:2000]

print(df.columns)
print(df.head())

Index(['X_jet', 'm', 'iphi', 'pt', 'ieta'], dtype='object')
                                               X_jet           m  iphi  \
0  [[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0...  291.988312  33.0   

           pt  ieta  
0  962.311523  16.0  


In [11]:
import torch
import numpy as np
from torch.utils.data import Dataset

class JetDataset(Dataset):
    def __init__(self, df):
        X_list = []
        y_list = []

        for i in range(len(df)):
            x_raw = df.iloc[i]['X_jet']

            channels = []

            for ch in x_raw:
                # 🔥 Flatten nested arrays safely
                flat = []

                for item in ch:
                    if isinstance(item, (list, np.ndarray)):
                        flat.extend(np.array(item).flatten())
                    else:
                        flat.append(item)

                flat = np.array(flat, dtype=np.float32)

                # Ensure size = 125
                flat = flat[:125]

                channels.append(flat)

            x = np.array(channels)  # (8,125)

            # Select required channels
            track_pt = x[0]
            ecal = x[3]

            # Create image
            img = np.zeros((2, 125, 125), dtype=np.float32)

            ieta = int(df.iloc[i]['ieta'])

            # Fill row
            img[0, ieta, :] = track_pt
            img[1, ieta, :] = ecal

            X_list.append(img)
            y_list.append(df.iloc[i]['m'])

        self.X = torch.tensor(np.array(X_list), dtype=torch.float32)
        self.y = torch.tensor(np.array(y_list), dtype=torch.float32)

# ✅ Normalize AFTER creating tensor
        self.y = (self.y - self.y.mean()) / self.y.std()

        print("Dataset Created ✅")
        print("X shape:", self.X.shape)
        print("y shape:", self.y.shape)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [12]:
dataset = JetDataset(df)

Dataset Created ✅
X shape: torch.Size([1, 2, 125, 125])
y shape: torch.Size([1])


/tmp/ipykernel_2883/2355974815.py:54: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  self.y = (self.y - self.y.mean()) / self.y.std()


In [14]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("top_gun_opendata_0.parquet.1")

print("Number of row groups:", parquet_file.num_row_groups)

Number of row groups: 150327


In [15]:
print("Total row groups:", parquet_file.num_row_groups)

Total row groups: 150327


In [25]:
import pyarrow.parquet as pq
import pandas as pd

parquet_file = pq.ParquetFile("top_gun_opendata_0.parquet.1")

dfs = []

for i in range(min(200, parquet_file.num_row_groups)):  # load many samples
    batch = parquet_file.read_row_group(i)
    dfs.append(batch.to_pandas())

df = pd.concat(dfs, ignore_index=True)

print("Total samples:", len(df))

Total samples: 200


In [26]:
dataset = JetDataset(df)

Dataset Created ✅
X shape: torch.Size([200, 2, 125, 125])
y shape: torch.Size([200])


In [27]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 160
Test: 40


In [21]:
import torch.nn as nn
import torch.nn.functional as F

class MassRegressor(nn.Module):
    def __init__(self):
        super(MassRegressor, self).__init__()

        self.conv1 = nn.Conv2d(2, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 15 * 15, 128)
        self.fc2 = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

In [28]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.MSELoss()   # for regression
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [29]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MassRegressor().to(device)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1).float()

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

Epoch [1/5] Loss: 0.9702
Epoch [2/5] Loss: 0.9702
Epoch [3/5] Loss: 0.9702
Epoch [4/5] Loss: 0.9702
Epoch [5/5] Loss: 0.9702


In [33]:
!pip install onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 13.6 MB/s eta 0:00:00


In [35]:
model.eval()

dummy_input = torch.randn(1, 2, 125, 125).to(device)

torch.onnx.export(
    model,
    dummy_input,
    "mass_model.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=18   # 🔥 use latest
)

print("Clean ONNX export ✅")

[torch.onnx] Obtain model graph for `MassRegressor([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MassRegressor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Clean ONNX export ✅


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


In [36]:
from google.colab import files

files.download("mass_model.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>